# Query Iceberg Tables via DuckDB + Nessie REST Catalog

Reads `demo.users`, `demo.transactions`, and Iceberg tables
written by the Flink job, using DuckDB's native Iceberg REST catalog support.

DuckDB `ATTACH`es directly to the Nessie Iceberg REST API — no metadata file
paths needed.

**Prerequisites:** the Docker Compose stack must be running (`docker compose up -d`).

In [1]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("INSTALL iceberg")
con.execute("LOAD httpfs")
con.execute("LOAD iceberg")

# S3 credentials — DuckDB uses these to sign its own S3 requests directly
con.execute("""
CREATE OR REPLACE SECRET minio (
    TYPE        s3,
    KEY_ID      'minioadmin',
    SECRET      'minioadmin',
    ENDPOINT    'minio:9000',
    URL_STYLE   'path',
    USE_SSL     false,
    REGION      'us-east-1'
)
""")

# ACCESS_DELEGATION_MODE 'none' tells DuckDB to skip Nessie's s3sign endpoint
# and sign S3 requests itself using the secret above
con.execute("""
ATTACH 'demo_lh' AS nessie (
    TYPE                   iceberg,
    ENDPOINT               'http://nessie:19120/iceberg',
    AUTHORIZATION_TYPE     'none',
    ACCESS_DELEGATION_MODE 'none'
)
""")

con.execute("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,nessie,demo,orders,[__],[UNKNOWN],False
1,nessie,demo,transactions,[__],[UNKNOWN],False
2,nessie,demo,users,[__],[UNKNOWN],False


---
## Users

In [2]:
# All users
con.execute("SELECT * FROM nessie.demo.users ORDER BY user_id").df()

,user_id,name,email,country,created_at
0,user-001,Rose Jones,rose.jones1@example.com,BR,2026-04-25 09:20:56.310
1,user-001,Alice Martinez,alice.martinez1@example.com,BR,2026-04-25 09:44:04.990
2,user-001,Hank Jones,hank.jones1@example.com,GB,2026-04-25 09:37:05.607
3,user-001,Grace Robinson,grace.robinson1@example.com,JP,2026-04-25 09:29:06.009
4,user-001,Karen Martin,karen.martin1@example.com,CA,2026-04-25 09:12:16.321
...,...,...,...,...,...
795,user-050,Leo Clark,leo.clark50@example.com,DE,2026-04-25 11:01:07.710
796,user-050,Dave Jones,dave.jones50@example.com,GB,2026-04-25 10:29:18.059
797,user-050,Alice Smith,alice.smith50@example.com,BR,2026-04-25 09:44:04.990
798,user-050,Olivia Smith,olivia.smith50@example.com,JP,2026-04-25 09:24:36.380


In [3]:
# Summary stats
con.execute("""
    SELECT
        count(*)                AS total_users,
        count(DISTINCT country) AS unique_countries,
        min(created_at)         AS earliest_signup,
        max(created_at)         AS latest_signup
    FROM nessie.demo.users
""").df()

,total_users,unique_countries,earliest_signup,latest_signup
0,800,8,2026-04-25 09:09:44.097,2026-04-25 11:14:08.926


In [4]:
# Users per country
con.execute("""
    SELECT
        country,
        count(*) AS user_count
    FROM nessie.demo.users
    GROUP BY country
    ORDER BY user_count DESC
""").df()

,country,user_count
0,US,111
1,FR,108
2,GB,108
3,AU,102
4,DE,96
5,JP,93
6,BR,91
7,CA,91


---
## Transactions

In [5]:
# Latest 20 transactions
con.execute("""
    SELECT *
    FROM nessie.demo.transactions
    ORDER BY event_time DESC
    LIMIT 20
""").df()

,transaction_id,user_id,amount,currency,type,status,event_time
0,txn-000221,user-037,1440.18,CAD,TRANSFER,COMPLETED,2026-04-25 11:17:50.311
1,txn-000220,user-044,219.69,CAD,DEPOSIT,REVERSED,2026-04-25 11:17:49.303
2,txn-000219,user-045,1072.12,USD,TRANSFER,PENDING,2026-04-25 11:17:48.299
3,txn-000218,user-048,746.08,GBP,TRANSFER,REVERSED,2026-04-25 11:17:47.293
4,txn-000217,user-045,119.57,EUR,PURCHASE,FAILED,2026-04-25 11:17:46.289
5,txn-000216,user-018,33.63,GBP,REFUND,REVERSED,2026-04-25 11:17:45.285
6,txn-000215,user-029,1466.90,GBP,PURCHASE,REVERSED,2026-04-25 11:17:44.281
7,txn-000214,user-024,1336.72,GBP,PURCHASE,COMPLETED,2026-04-25 11:17:43.270
8,txn-000213,user-023,1810.68,CAD,DEPOSIT,PENDING,2026-04-25 11:17:42.264
9,txn-000212,user-041,1295.78,CAD,PURCHASE,REVERSED,2026-04-25 11:17:41.253


In [6]:
# Summary stats
con.execute("""
    SELECT
        count(*)                AS total_transactions,
        count(DISTINCT user_id) AS unique_users,
        round(sum(amount), 2)   AS total_volume,
        round(avg(amount), 2)   AS avg_amount,
        min(event_time)         AS earliest,
        max(event_time)         AS latest
    FROM nessie.demo.transactions
""").df()

,total_transactions,unique_users,total_volume,avg_amount,earliest,latest
0,7281,50,7248430.07,995.53,2026-04-25 09:09:44.438,2026-04-25 11:17:50.311


In [7]:
# Volume by transaction type
con.execute("""
    SELECT
        type,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM nessie.demo.transactions
    GROUP BY type
    ORDER BY tx_count DESC
""").df()

,type,tx_count,total_amount
0,PURCHASE,1496,1456758.55
1,DEPOSIT,1461,1441557.21
2,TRANSFER,1460,1461721.65
3,REFUND,1449,1461320.24
4,WITHDRAWAL,1415,1427072.42


In [8]:
# Breakdown by status
con.execute("""
    SELECT
        status,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM nessie.demo.transactions
    GROUP BY status
    ORDER BY tx_count DESC
""").df()

,status,tx_count,total_amount
0,PENDING,1898,1896179.72
1,REVERSED,1804,1758367.86
2,COMPLETED,1797,1833332.38
3,FAILED,1782,1760550.11


In [10]:
# Volume by currency
con.execute("""
    SELECT
        currency,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM nessie.demo.transactions
    GROUP BY currency
    ORDER BY total_amount DESC
""").df()

,currency,tx_count,total_amount
0,USD,1542,1545124.08
1,EUR,1431,1461009.53
2,AUD,1434,1418031.28
3,GBP,1429,1415862.08
4,CAD,1445,1408403.10


In [11]:
# Top 10 users by transaction volume
con.execute("""
    SELECT
        user_id,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_spent
    FROM nessie.demo.transactions
    GROUP BY user_id
    ORDER BY total_spent DESC
    LIMIT 10
""").df()

,user_id,tx_count,total_spent
0,user-045,172,174542.62
1,user-047,165,174056.28
2,user-042,166,171010.87
3,user-028,170,169821.20
4,user-008,158,165691.17
5,user-026,157,164610.27
6,user-018,147,164248.12
7,user-017,165,163679.42
8,user-007,158,159468.11
9,user-043,154,156259.87


---
## Cross-table: join users ↔ transactions

In [16]:
# Enrich transactions with user name and country
con.execute("""
    SELECT
        t.transaction_id,
        u.name,
        u.country,
        t.type,
        t.currency,
        t.amount,
        t.status,
        t.event_time
    FROM nessie.demo.transactions AS t
    JOIN nessie.demo.users        AS u ON t.user_id = u.user_id
    ORDER BY t.event_time DESC
    LIMIT 20
""").df()

,transaction_id,name,country,type,currency,amount,status,event_time
0,txn-000251,Bob Davies,GB,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
1,txn-000251,Alice Garcia,AU,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
2,txn-000251,Quinn Jones,GB,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
3,txn-000251,Olivia Davies,AU,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
4,txn-000251,Olivia Lewis,US,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
5,txn-000251,Carol Robinson,JP,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
6,txn-000251,Mia Roberts,GB,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
7,txn-000251,Mia Wilson,CA,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
8,txn-000251,Leo Johnson,CA,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586
9,txn-000251,Leo Thomas,AU,PURCHASE,CAD,715.0,REVERSED,2026-04-25 11:18:20.586


In [17]:
# Total transaction volume per country
con.execute("""
    SELECT
        u.country,
        count(*)                AS tx_count,
        round(sum(t.amount), 2) AS total_volume
    FROM nessie.demo.transactions AS t
    JOIN nessie.demo.users        AS u ON t.user_id = u.user_id
    GROUP BY u.country
    ORDER BY total_volume DESC
""").df()

,country,tx_count,total_volume
0,US,16362,16377861.79
1,FR,15785,15711864.15
2,GB,15684,15627229.21
3,AU,14889,14722256.11
4,DE,14122,14133229.01
5,JP,13747,13603745.70
6,BR,13298,13193279.45
7,CA,13089,13038336.82


---
## Snapshot history

In [18]:
# Snapshot history is exposed via the Nessie API
import urllib.request, json as _json

for table in ("users", "transactions"):
    url = f"http://nessie:19120/api/v1/contents/demo.{table}"
    with urllib.request.urlopen(url) as r:
        data = _json.loads(r.read())
    print(f"=== {table} === snapshotId={data['snapshotId']}  metadata={data['metadataLocation']}")


=== users === snapshotId=8286689221806640123  metadata=s3://iceberg-warehouse/demo/users_e0c93d07-499b-49e1-b9d1-02747444548d/metadata/00000-553fc4da-f5ad-4d52-a590-00de26b692c4.metadata.json
=== transactions === snapshotId=6669932774363812477  metadata=s3://iceberg-warehouse/demo/transactions_77a19506-19d2-4788-a5cb-1a318a0c5cfb/metadata/00000-d94acf00-10df-456b-ad88-9d60acc8a638.metadata.json
=== orders === snapshotId=2611875991924464532  metadata=s3://iceberg-warehouse/demo/orders_4ec29bba-7f58-4f4a-9b16-dc6e27beefbd/metadata/00000-1e46daf7-d1cd-4d24-b3c2-126869718e38.metadata.json
